# B = 63, 64, 65 on one GPU

Three contiguous batches at five layers, so the vector width walks all
three levels inside one lattice:

| B | v2(B) | v2(elements) | width |
|---|---|---|---|
| 63 | 0 | 0 | 1 |
| **64** | **6** | **6** | **4** |
| 65 | 0 | 0 | 1 |

B = 64 is **bracketed** by two scalar neighbours, so neither comparison
can be blamed on one unlucky odd batch -- which is what happened on the
H100, where B = 65 turned out to sit on a bump of the broad curve and
understated the effect 3.5x.

Arms are interleaved (63, 64, 65, 63, 64, 65, ...) so a GPU that warms
up spreads the drift across all three instead of loading it onto one.
Compilation runs before the timer.

**On the units.** The headline metric is microseconds per replica *per
sweep*. To get a wall clock, multiply by the replica count AND by the
sweep count: 60,000 sweeps here. The report at the end prints both.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi",
  "--query-gpu=name,memory.total,clocks.max.sm,driver_version",
  "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

In [ ]:
import sys, subprocess
def gpu_ok():
    try:
        import jax
        return any(d.platform == "gpu" for d in jax.devices())
    except Exception:
        return False
if not gpu_ok():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "jax[cuda12]"], check=True)
    for m in [k for k in list(sys.modules) if k.startswith("jax")]:
        del sys.modules[m]
import jax
print("jax", jax.__version__, jax.devices())

In [ ]:
# sim_core.py -- byte-identical to the repository copy. sha256 840b68448e8391ae9b07aa7a1161cb9d36cef67667489288756d3c044a66d058
import hashlib, io
_src = '"""Monte Carlo engine for the nanodisk, faithful to the project\'s reference notebook.\n\nTaken from `00_benchmark_mc_observables.ipynb` -- the same checkerboard, the same\nuniform-sphere proposal, and in particular the same DMI: **in-plane bonds only\n(x and y), with a minus sign**. That is interfacial DMI, the physics of a thin\nfilm whose DM vector comes from an interface, and it is not interchangeable with\nthe all-axes form: on the same configuration the total energy differs by 50%.\n\nTwo terms the reference notebook does not carry are added here because the\ntutor\'s dataset varies them:\n\n- `KanS`, the surface anisotropy, applied where a site has fewer than six\n  in-disk nearest neighbours (the coordination test used in the group\'s\n  multi-GPU notebooks).\n- `Kan2`, read as the SECOND-ORDER constant of the cubic anisotropy expansion,\n\n      E_an = (Kan1 / So^4)(sx^2 sy^2 + sx^2 sz^2 + sy^2 sz^2)\n           + (Kan2 / So^6)(sx^2 sy^2 sz^2)\n\n  which is the textbook K1/K2 series and matches the naming. It is a single-site\n  term, so unlike a second-neighbour coupling it leaves the checkerboard exact.\n"""\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nfrom jax import random\n\n\ndef build_geometry(Rd: float = 18.30, L: int = 5):\n    """Disk mask, checkerboard sublattices and the surface mask."""\n    L2 = int(Rd) + 1\n    N = 2 * L2 + 1\n    x = np.arange(-L2, L2 + 1)\n    xg, yg = np.meshgrid(x, x, indexing="xy")\n    disk2d = (xg ** 2 + yg ** 2) <= Rd ** 2\n    disco = np.broadcast_to(disk2d, (L, N, N)).copy()\n\n    zc, yc, xc = np.indices((L, N, N))\n    checker = (xc + yc + zc) % 2\n    even = (checker == 0) & disco\n    odd = (checker == 1) & disco\n\n    # Surface = fewer than six in-disk nearest neighbours. x and y wrap, z does\n    # not, so the top and bottom layers are surface everywhere.\n    surface = np.zeros_like(disco, dtype=bool)\n    for z in range(L):\n        for y in range(N):\n            for xx in range(N):\n                if not disco[z, y, xx]:\n                    continue\n                n = 0\n                for dz, dy, dx in ((0, 0, 1), (0, 0, -1), (0, 1, 0),\n                                   (0, -1, 0), (1, 0, 0), (-1, 0, 0)):\n                    nz, ny, nx = z + dz, (y + dy) % N, (xx + dx) % N\n                    if 0 <= nz < L and disco[nz, ny, nx]:\n                        n += 1\n                if n < 6:\n                    surface[z, y, xx] = True\n    return dict(L=L, N=N, Rd=Rd, disco=disco, even=even, odd=odd,\n                surface=surface, disk2d=disk2d, n_active=int(disco.sum()),\n                n_surface=int(surface.sum()))\n\n\ndef build_engine(geo, So=1.0):\n    """Returns (substep, energy, magnetization, topological_charge, anneal)."""\n    L, N = geo["L"], geo["N"]\n    even_j = jnp.asarray(geo["even"][None, ..., None])\n    odd_j = jnp.asarray(geo["odd"][None, ..., None])\n    disco_j = jnp.asarray(geo["disco"][None, ..., None])\n    surf_j = jnp.asarray(geo["surface"][None, ...])\n    disk2d_j = jnp.asarray(geo["disk2d"])\n    n_act = geo["n_active"]\n\n    def _an(s, Kan1, Kan2, K1_eff):\n        """Cubic anisotropy: the K1 invariant plus the K2 (sixth-order) one."""\n        sx, sy, sz = s[..., 0], s[..., 1], s[..., 2]\n        e = (K1_eff / So ** 4) * ((sx * sy) ** 2 + (sx * sz) ** 2 + (sy * sz) ** 2)\n        return e + (Kan2 / So ** 6) * (sx * sy * sz) ** 2\n\n    def substep(spins, key, beta, mask, p, K1_eff):\n        S_xp = jnp.roll(spins, -1, 3); S_xm = jnp.roll(spins, 1, 3)\n        S_yp = jnp.roll(spins, -1, 2); S_ym = jnp.roll(spins, 1, 2)\n        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)\n        S_zm = jnp.roll(spins, 1, 1).at[:, 0].set(0.0)\n\n        kt, ka = random.split(key)\n        tr = random.normal(kt, spins.shape)\n        tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True) * So\n        tr = jnp.where(mask, tr, spins)\n        dS = tr - spins\n\n        sv = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm\n        dE = -p["Jex"] * jnp.sum(dS * sv, axis=-1)\n        dE = dE + _an(tr, p["Kan1"], p["Kan2"], K1_eff) - _an(spins, p["Kan1"], p["Kan2"], K1_eff)\n        # Interfacial DMI: in-plane bonds only, minus sign. See module docstring.\n        dE = dE - p["KDM"] * (dS[..., 1] * (S_xp[..., 2] - S_xm[..., 2])\n                              - dS[..., 2] * (S_xp[..., 1] - S_xm[..., 1])\n                              + dS[..., 2] * (S_yp[..., 0] - S_ym[..., 0])\n                              - dS[..., 0] * (S_yp[..., 2] - S_ym[..., 2]))\n        dE = dE - p["Hex"] * (dS[..., 0] * jnp.cos(p["g"]) + dS[..., 2] * jnp.sin(p["g"]))\n\n        r = random.uniform(ka, dE.shape)\n        acc = ((dE < 0) | (r < jnp.exp(-dE * beta)))[..., None] & mask\n        return jnp.where(acc, tr, spins)\n\n    def energy(spins, p, K1_eff):\n        """Total energy; each bond counted once via the +x, +y, +z shifts only."""\n        S_xp = jnp.roll(spins, -1, 3)\n        S_yp = jnp.roll(spins, -1, 2)\n        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)\n        E = -p["Jex"] * jnp.sum(spins * (S_xp + S_yp + S_zp), axis=-1)\n        sx, sy, sz = spins[..., 0], spins[..., 1], spins[..., 2]\n        E = E + _an(spins, p["Kan1"], p["Kan2"], K1_eff)\n        E = E - p["KDM"] * (sy * S_xp[..., 2] - sz * S_xp[..., 1]\n                            + sz * S_yp[..., 0] - sx * S_yp[..., 2])\n        E = E - p["Hex"] * (sx * jnp.cos(p["g"]) + sz * jnp.sin(p["g"]))\n        return jnp.sum(E * disco_j[..., 0], axis=(-3, -2, -1))\n\n    def magnetization(spins):\n        M = jnp.sum(spins * disco_j, axis=(1, 2, 3)) / n_act\n        return jnp.linalg.norm(M, axis=-1), M[:, 2]\n\n    def _omega(s1, s2, s3):\n        num = jnp.sum(s1 * jnp.cross(s2, s3), axis=-1)\n        den = (1.0 + jnp.sum(s1 * s2, -1) + jnp.sum(s2 * s3, -1) + jnp.sum(s3 * s1, -1))\n        return 2.0 * jnp.arctan2(num, den)\n\n    def topo_charge(spins):\n        """Berg-Luscher charge of every layer: quantised, unlike a finite difference."""\n        s00 = spins[..., :-1, :-1, :]; s10 = spins[..., :-1, 1:, :]\n        s01 = spins[..., 1:, :-1, :];  s11 = spins[..., 1:, 1:, :]\n        valid = (disk2d_j[:-1, :-1] & disk2d_j[:-1, 1:]\n                 & disk2d_j[1:, :-1] & disk2d_j[1:, 1:])\n        w = _omega(s00, s10, s11) + _omega(s00, s11, s01)\n        return jnp.sum(jnp.where(valid, w, 0.0), axis=(-2, -1)) / (4.0 * jnp.pi)\n\n    def anneal(spins, key, betas, p, n_therm, n_meas):\n        K1_eff = jnp.where(surf_j, p["KanS"], p["Kan1"])\n\n        def per_T(carry, beta):\n            s, k = carry\n\n            def therm(_, v):\n                s, k = v\n                k, k1, k2 = random.split(k, 3)\n                s = substep(s, k1, beta, even_j, p, K1_eff)\n                s = substep(s, k2, beta, odd_j, p, K1_eff)\n                return s, k\n            s, k = jax.lax.fori_loop(0, n_therm, therm, (s, k))\n\n            b = s.shape[0]\n            z0 = jnp.zeros(b)\n\n            # C_v and chi are variances, and accumulating raw E and E^2\n            # destroys them in float32. At T = 0.2 this run has <E> ~ -1.7e4, so\n            # <E^2> ~ 2.9e8, where one float32 ulp is ~34 -- larger than the\n            # variance itself. <E^2> - <E>^2 then comes out NEGATIVE and C_v\n            # reads -437: not a small error but pure cancellation.\n            #\n            # Measuring the fluctuation about a reference taken once, before the\n            # loop, keeps every accumulated quantity at the scale of the\n            # fluctuation rather than of the total. A constant shift leaves the\n            # variance unchanged, so this is exact. Both the centred moments and\n            # the reference are returned, and `reduce_obs` recombines them.\n            E_ref = energy(s, p, K1_eff)\n            M_ref, _ = magnetization(s)\n\n            def meas(_, v):\n                s, k, sE, sE2, sM, sM2, sMz, sQ = v\n                k, k1, k2 = random.split(k, 3)\n                s = substep(s, k1, beta, even_j, p, K1_eff)\n                s = substep(s, k2, beta, odd_j, p, K1_eff)\n                dE = energy(s, p, K1_eff) - E_ref\n                M, Mz = magnetization(s)\n                dM = M - M_ref\n                Q = topo_charge(s)[:, L // 2]\n                return (s, k, sE + dE, sE2 + dE * dE,\n                        sM + dM, sM2 + dM * dM, sMz + Mz, sQ + Q)\n\n            s, k, sE, sE2, sM, sM2, sMz, sQ = jax.lax.fori_loop(\n                0, n_meas, meas, (s, k, z0, z0, z0, z0, z0, z0))\n            obs = jnp.stack([sE / n_meas, sE2 / n_meas,\n                             sM / n_meas, sM2 / n_meas,\n                             sMz / n_meas, sQ / n_meas, E_ref, M_ref])\n            # One middle-layer snapshot per temperature, replica 0 only. The\n            # whole batch would be 30 x 100 x 5 x 39 x 39 x 3 floats (~2.7 GB);\n            # one replica\'s middle layer is 182 kB and is what the state-\n            # transition figure needs.\n            snap = s[0, L // 2, :, :, 2]\n            return (s, k), (obs, snap)\n\n        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (spins, key), betas)\n        return all_obs, s, snaps\n\n    return dict(substep=substep, energy=energy, magnetization=magnetization,\n                topo_charge=topo_charge, anneal=anneal)\n\n\ndef init_spins(key, geo, batch, So=1.0):\n    L, N = geo["L"], geo["N"]\n    x = random.normal(key, (batch, L, N, N, 3))\n    x = x / jnp.linalg.norm(x, axis=-1, keepdims=True) * So\n    return x * jnp.asarray(geo["disco"][None, ..., None])\n\n\ndef reduce_obs(obs, temps, n_spins, kB=0.086173404):\n    """Accumulators -> per-temperature scalars.\n\n    `lax.scan` stacks each step\'s output along a NEW LEADING axis, so `anneal`\n    returns `(n_T, 6, batch)` and the observable index is the SECOND axis. The\n    obvious `obs[i]` indexes temperatures instead, silently mixing the two --\n    and a test with six temperatures and six observables cannot tell the\n    difference, since both shapes read `(6, 6, batch)`.\n    """\n    obs = np.asarray(obs)\n    assert obs.ndim == 3 and obs.shape[1] == 8, (\n        f"expected (n_T, 8, batch) from lax.scan, got {obs.shape}")\n    dE, dE2, dM, dM2, Mz, Q, E_ref, M_ref = [obs[:, i] for i in range(8)]\n    T = np.asarray(temps)[:, None]\n    # Variances of the CENTRED quantities: both terms sit at the scale of the\n    # fluctuation, so there is nothing to cancel.\n    var_E = dE2 - dE ** 2\n    var_M = dM2 - dM ** 2\n    E = dE + E_ref\n    M = dM + M_ref\n    Cv = var_E / (kB * T ** 2 * n_spins)\n    chi = var_M / (kB * T)\n    return dict(T=np.asarray(temps),\n                E=E.mean(1) / n_spins, E_sd=E.std(1) / n_spins,\n                M=M.mean(1), M_sd=M.std(1),\n                Mz=Mz.mean(1), Cv=Cv.mean(1), chi=chi.mean(1),\n                Q=Q.mean(1), Q_sd=Q.std(1))\n'
io.open("sim_core.py", "w", encoding="utf-8").write(_src)
assert hashlib.sha256(_src.encode()).hexdigest() == "840b68448e8391ae9b07aa7a1161cb9d36cef67667489288756d3c044a66d058", "tampered"
print("sim_core.py: sha256 verified")

In [ ]:
# pair_test.py -- byte-identical to the repository copy. sha256 8075b606cffc3e4510aed7d2952ee8005891f1ced6682a4a8515fef076d95cd5
import hashlib, io
_src = '"""B = 64 against B = 65: the smallest experiment that shows the rule.\n\nOne replica apart, and the whole effect:\n\n    elements = B * L * N^2 * 3,  N odd, so  v2(elements) = v2(B) + v2(L)\n    XLA gives each thread  min(4, 2^v2(elements))  elements\n\n    L = 5, B = 64:  v2(5) + v2(64) = 0 + 6 = 6  ->  width 4  ->  penalty\n    L = 5, B = 65:  v2(5) + v2(65) = 0 + 0 = 0  ->  width 1  ->  free\n\nEverything else is held: the same lattice, the same schedule, the same seeds,\nthe same container, the same physical GPU. The only difference is one replica.\n\nTwo design points that matter for believing the result.\n\nFirst, the two arms are **interleaved** -- 64, 65, 64, 65, ... -- not run as two\nblocks. A GPU that warms up or gets throttled during the run would bias whole\nblocks; interleaving spreads any drift across both arms instead of loading it\nonto one.\n\nSecond, compilation is **outside** the timer. `.lower().compile()` runs before\nthe clock starts, so the number is execution only. That matters here because the\ntwo shapes compile to different programs, and including compile time would\nmeasure the compiler rather than the kernel.\n\nRuns unchanged on Modal and on Kaggle: it imports nothing but jax, numpy and\n`sim_core`, and takes the device as given.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport time\n\n\ndef run_pair(batches=(64, 65), repeats=3, Rd=18.30, L=5, n_temps=40,\n             n_therm=1000, n_meas=500, seed=11, out=None, label=""):\n    """Time each batch `repeats` times, interleaved, and report the contrast."""\n    import numpy as np\n    import jax\n    import jax.numpy as jnp\n    import sim_core as S\n\n    def v2(n):\n        k = 0\n        while n % 2 == 0:\n            n //= 2\n            k += 1\n        return k\n\n    geo = S.build_geometry(Rd=float(Rd), L=int(L))\n    eng = S.build_engine(geo)\n    N = int(geo["N"])\n    n_active = int(np.asarray(geo["disco"]).sum())\n    kB = 0.086173404\n    betas = jnp.asarray(1.0 / (kB * np.linspace(20.0, 0.1, int(n_temps))))\n    sweeps = int(n_temps) * (int(n_therm) + int(n_meas))\n\n    # One theta broadcast to every replica. `sim_core` wants per-chain columns,\n    # so each parameter is a (B, 1, 1, 1) array rather than a scalar.\n    theta = dict(Jex=1.0, Kan1=0.0012, Kan2=0.0, KanS=0.0012,\n                 Hex=0.0, KDM=0.0, g=float(np.deg2rad(90.0)))\n\n    dev = jax.devices()[0]\n    print(f"device: {dev.device_kind}  ({dev.platform})", flush=True)\n    print(f"lattice: Rd={Rd} L={L} N={N} n_active={n_active}  "\n          f"stride={12 * L * N * N:,} B/replica", flush=True)\n    print(f"schedule: {n_temps} temperatures x {n_therm + n_meas} sweeps "\n          f"= {sweeps:,} sweeps per cell, compilation excluded", flush=True)\n    for B in batches:\n        ve = v2(B) + v2(int(L))\n        print(f"  B={B:4d}  v2({B})={v2(B)} + v2(L={L})={v2(int(L))} -> "\n              f"v2(elements)={ve}  width={min(4, 2 ** ve)}"\n              f"{\'   <-- predicted penalty\' if ve >= 2 else \'\'}", flush=True)\n\n    def body(sp, kk, p):\n        return eng["anneal"](sp, kk, betas, p, int(n_therm), int(n_meas))\n\n    built = {}\n    for B in batches:\n        cols = {k: jnp.asarray(np.full((B, 1, 1, 1), v, dtype=np.float32))\n                for k, v in theta.items()}\n        args = (S.init_spins(jax.random.PRNGKey(seed), geo, B),\n                jax.random.PRNGKey(seed + 1), cols)\n        built[B] = (jax.jit(body).lower(*args).compile(), args)\n        print(f"  compiled B={B}", flush=True)\n\n    # Interleaved: one repeat of every batch, then the next repeat.\n    samples = {B: [] for B in batches}\n    for r in range(int(repeats)):\n        for B in batches:\n            fn, args = built[B]\n            t0 = time.perf_counter()\n            jax.block_until_ready(fn(*args))\n            dt = time.perf_counter() - t0\n            samples[B].append(dt)\n            print(f"  round {r + 1}/{repeats}  B={B:4d}  {dt:7.2f} s  "\n                  f"{dt * 1e6 / (sweeps * B):7.3f} us/replica-sweep", flush=True)\n\n    rows = []\n    for B in batches:\n        a = np.array(samples[B])\n        us = a * 1e6 / (sweeps * B)\n        rows.append(dict(B=int(B), v2_elements=v2(B) + v2(int(L)),\n                         width=min(4, 2 ** (v2(B) + v2(int(L)))),\n                         wall_s=[float(x) for x in a],\n                         us_per_replica_sweep=[float(x) for x in us],\n                         us_mean=float(us.mean()), us_sd=float(us.std()),\n                         us_min=float(us.min()), us_max=float(us.max())))\n\n    wide = [r for r in rows if r["width"] == 4]\n    narrow = [r for r in rows if r["width"] < 4]\n    pen = overlap = None\n    if wide and narrow:\n        w = float(np.mean([r["us_mean"] for r in wide]))\n        n = float(np.mean([r["us_mean"] for r in narrow]))\n        pen = w / n - 1\n        overlap = min(min(r["us_per_replica_sweep"]) for r in wide) <= \\\n            max(max(r["us_per_replica_sweep"]) for r in narrow)\n\n    print("\\n=== result ===", flush=True)\n    for r in rows:\n        print(f"  B={r[\'B\']:4d} width {r[\'width\']}  "\n              f"{r[\'us_mean\']:7.3f} +/- {r[\'us_sd\']:.3f} us  "\n              f"(min {r[\'us_min\']:.3f}, max {r[\'us_max\']:.3f})", flush=True)\n    if pen is not None:\n        print(f"  => width-4 penalty {pen:+.1%}   "\n              f"overlap between the two samples: "\n              f"{\'YES\' if overlap else \'NONE\'}", flush=True)\n\n    res = dict(label=label, rows=rows, penalty=pen, overlap=overlap,\n               device=dev.device_kind, platform=dev.platform,\n               Rd=float(Rd), L=int(L), N=N, n_active=n_active,\n               stride_bytes=12 * int(L) * N * N, sweeps_per_cell=sweeps,\n               n_temps=int(n_temps), n_therm=int(n_therm), n_meas=int(n_meas),\n               repeats=int(repeats), seed=int(seed), interleaved=True,\n               aot_compiled=True, jax_version=jax.__version__)\n    if out:\n        with open(out, "w") as f:\n            json.dump(res, f, indent=1)\n        print(f"\\nwrote {out}", flush=True)\n    return res\n'
io.open("pair_test.py", "w", encoding="utf-8").write(_src)
assert hashlib.sha256(_src.encode()).hexdigest() == "8075b606cffc3e4510aed7d2952ee8005891f1ced6682a4a8515fef076d95cd5", "tampered"
print("pair_test.py: sha256 verified")

In [ ]:
import sys, json
sys.path.insert(0, ".")
import pair_test
res = pair_test.run_pair(batches=(63, 64, 65), repeats=3, Rd=18.30, L=5,
                         n_temps=40, n_therm=1000, n_meas=500, seed=11,
                         out="/kaggle/working/trio.json",
                         label="kaggle-trio")

In [ ]:
by = {r["B"]: r for r in res["rows"]}
print()
print("B    width   us/replica-sweep   wall per cell")
for r in res["rows"]:
    w = sum(r["wall_s"]) / len(r["wall_s"])
    print("%-4d %5d %16.3f %13.1f s" % (r["B"], r["width"], r["us_mean"], w))
print()
# B=64 is bracketed, so report it against BOTH scalar neighbours.
for lo in (63, 65):
    a = sum(by[64]["wall_s"]) / 3
    b = sum(by[lo]["wall_s"]) / 3
    pct = 100 * (by[64]["us_mean"] / by[lo]["us_mean"] - 1)
    print("B=64 vs B=%d:  %.1f s vs %.1f s  ->  %+.1f s  (%+.1f%% per replica-sweep)"
          % (lo, a, b, a - b, pct))
print()
print("A production run is 200 temperatures x 20,000 sweeps = 4,000,000.")
P = 200 * 20000
for r in res["rows"]:
    h = r["us_mean"] * r["B"] * P / 1e6 / 3600
    print("  B=%-4d %6.2f GPU-hours" % (r["B"], h))